1 Design a strategy

physical distances are known, the ruler divisions are calibrated - this is the ground truth. 
CamIm03 is x100, so its pixel size will be the smallest one. 
Dark features on bright background 
Uneven illumination - correct before segmentation - binary morphology - Black Top hat: closing(f, SE) - f - removes the background, leaving only the dark ruler lines. 
SE needs to be larger than the widest ruler line - then SE cannot fit inside dark lines, rolls over them and only sees background. 



In [1]:
import diplib as dip
import numpy as np
import matplotlib.pyplot as plt

DIPlib -- a quantitative image analysis library
Version 3.6.0 (Oct 23 2025)
For more information see https://diplib.org


Q1

In [2]:
images = {
    "CamIm01": "CamIm01.tif",
    "CamIm02": "CamIm02.tif",
    "CamIm03": "CamIm03.tif",
}

se_size  = 100
se_shape = "elliptic"
se = dip.SE(se_size, se_shape)

fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for row, (name, path) in enumerate(images.items()):

    img = dip.ImageRead(path)
    img = dip.Convert(img, "SFLOAT")

    img_extended = dip.ExtendImage(img, [se_size, se_size],
                                    boundaryCondition=["mirror"])

    closing_ext = dip.Closing(img_extended, se)
    tophat_ext  = closing_ext - img_extended

    w, h = img.Sizes()
    corrected = tophat_ext[se_size:se_size+w, se_size:se_size+h]

    binary, threshold = dip.Threshold(corrected, method="otsu")
    print(f"{name} - Otsu threshold: {threshold:.2f}")

    binary = dip.Closing(binary, dip.SE(5, "rectangular"))

    axes[row, 0].imshow(np.array(img),                          cmap="gray")
    axes[row, 0].set_title(f"{name} - Original")
    axes[row, 1].imshow(np.array(dip.Closing(img, se)),         cmap="gray")
    axes[row, 1].set_title(f"{name} - Closing")
    axes[row, 2].imshow(np.array(corrected),                    cmap="gray")
    axes[row, 2].set_title(f"{name} - Black Hat (corrected)")
    axes[row, 3].imshow(np.array(binary),                       cmap="gray")
    axes[row, 3].set_title(f"{name} - Binary (Otsu t={threshold:.1f})")

    for ax in axes[row]:
        ax.axis("off")

    # ── Save separate figure for each image ───────────────────
    fig_sep, axes_sep = plt.subplots(1, 4, figsize=(20, 5))
    axes_sep[0].imshow(np.array(img),                          cmap="gray")
    axes_sep[0].set_title("Original")
    axes_sep[1].imshow(np.array(dip.Closing(img, se)),         cmap="gray")
    axes_sep[1].set_title("Closing")
    axes_sep[2].imshow(np.array(corrected),                    cmap="gray")
    axes_sep[2].set_title("Black Hat (corrected)")
    axes_sep[3].imshow(np.array(binary),                       cmap="gray")
    axes_sep[3].set_title(f"Binary (Otsu t={threshold:.1f})")

    for ax in axes_sep:
        ax.axis("off")

    plt.suptitle(f"{name} - Calibration Pipeline (SE size={se_size}, elliptic)",
                    fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{name}_result.png", dpi=150)
    plt.close()
    print(f"Saved: {name}_result.png")

plt.suptitle("Part 3.1 - Calibration Pipeline (SE size=100, elliptic)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("part31_all_images.png", dpi=150)
plt.close()
print("Saved: part31_all_images.png")

CamIm01 - Otsu threshold: 49.02
Saved: CamIm01_result.png
CamIm02 - Otsu threshold: 62.00
Saved: CamIm02_result.png
CamIm03 - Otsu threshold: 64.02
Saved: CamIm03_result.png
Saved: part31_all_images.png


Measurement:
1. label the connected components
2. filter by shape (a priori knowledge)
3. use center measurement to get vertical position of each detected line
4. compute gaps. 

In [3]:
MAJOR_SPACING_MM  = 0.1
REF_IMAGE         = "CamIm03"
REF_MAGNIFICATION = 100
STANDARD_MAGS     = [4, 5, 10, 20, 40, 63, 100]

import json

se_size  = 100
se_shape = "elliptic"
se       = dip.SE(se_size, se_shape)

results = {}

for name, path in images.items():

    # ── Reproduce binary from previous step ──────────────────
    img = dip.ImageRead(path)
    img = dip.Convert(img, "SFLOAT")

    img_extended = dip.ExtendImage(img, [se_size, se_size],
                                    boundaryCondition=["mirror"])
    closing_ext  = dip.Closing(img_extended, se)
    tophat_ext   = closing_ext - img_extended

    w, h      = img.Sizes()
    corrected = tophat_ext[se_size:se_size+w, se_size:se_size+h]

    binary, threshold = dip.Threshold(corrected, method="otsu")
    binary = dip.Closing(binary, dip.SE(5, "rectangular"))

    # ── Step 1: Label connected components ───────────────────
    labeled = dip.Label(binary)

    # ── Step 2: Measure Feret length and centroid ─────────────
    msr = dip.MeasurementTool.Measure(
              labeled, corrected,
              features=["Feret", "Center"])

    objects   = list(msr.Objects())
    feret_max = np.array([msr[o]["Feret"][0] for o in objects])
    centers_y = np.array([msr[o]["Center"][1] for o in objects])

    # ── Step 3: Keep only long lines (major lines > 5% image width) ──
    long_mask = feret_max > 0.05 * w
    cy_sorted = np.sort(centers_y[long_mask])

    # ── Step 4: Compute gaps, filter minor-line gaps ──────────
    diffs         = np.diff(cy_sorted)
    gap_threshold = np.max(diffs) * 0.5
    major_diffs   = diffs[diffs >= gap_threshold]
    if len(major_diffs) == 0:
        major_diffs = diffs

    median_gap_px = np.median(major_diffs)

    # ── Step 5: Pixel size ────────────────────────────────────
    pixel_size_mm = MAJOR_SPACING_MM / median_gap_px
    pixel_size_um = pixel_size_mm * 1000

    results[name] = {
        "pixel_size_mm":  pixel_size_mm,
        "pixel_size_um":  pixel_size_um,
        "median_gap_px":  median_gap_px,
        "n_lines":        len(cy_sorted),
        "line_centers_y": cy_sorted.tolist(),
    }

    print(f"{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  Lines detected      : {len(cy_sorted)}")
    print(f"  Line centers y (px) : {np.round(cy_sorted, 1)}")
    print(f"  All gaps (px)       : {np.round(diffs, 2)}")
    print(f"  Major gaps (px)     : {np.round(major_diffs, 2)}")
    print(f"  Median major gap    : {median_gap_px:.2f} px")
    print(f"  Pixel size          : {pixel_size_mm:.6f} mm/px")
    print(f"  Pixel size          : {pixel_size_um:.4f} µm/px")
    print()

# ── Table 1: Pixel sizes ──────────────────────────────────────
print("=" * 52)
print(f"{'Table 1: Pixel Sizes':^52}")
print("=" * 52)
print(f"{'Image':<10} {'Gap (px)':>12} {'mm/px':>14} {'µm/px':>14}")
print("-" * 50)
for name, r in results.items():
    print(f"{name:<10} {r['median_gap_px']:>12.2f} "
          f"{r['pixel_size_mm']:>14.6f} {r['pixel_size_um']:>14.4f}")

# ── Table 2: Magnifications ───────────────────────────────────
ref_px = results[REF_IMAGE]["pixel_size_mm"]

print()
print("=" * 56)
print(f"{'Table 2: Derived Magnifications (CamIm03 = 100x)':^56}")
print("=" * 56)
print(f"{'Image':<10} {'µm/px':>12} {'Computed mag':>16} {'Probable mag':>14}")
print("-" * 52)
for name, r in results.items():
    comp_mag = REF_MAGNIFICATION * (ref_px / r["pixel_size_mm"])
    prob_mag = min(STANDARD_MAGS, key=lambda m: abs(m - comp_mag))
    results[name]["computed_mag"] = comp_mag
    results[name]["probable_mag"] = prob_mag
    print(f"{name:<10} {r['pixel_size_um']:>12.4f} "
          f"{comp_mag:>14.1f}x {prob_mag:>12}x")

# Save results to file
with open('calibration_results.txt', 'w') as f:
    f.write("=" * 52 + "\n")
    f.write(f"{'Table 1: Pixel Sizes':^52}\n")
    f.write("=" * 52 + "\n")
    f.write(f"{'Image':<10} {'Gap (px)':>12} {'mm/px':>14} {'µm/px':>14}\n")
    f.write("-" * 50 + "\n")
    for name, r in results.items():
        f.write(f"{name:<10} {r['median_gap_px']:>12.2f} "
              f"{r['pixel_size_mm']:>14.6f} {r['pixel_size_um']:>14.4f}\n")
    f.write("\n")
    f.write("=" * 56 + "\n")
    f.write(f"{'Table 2: Derived Magnifications (CamIm03 = 100x)':^56}\n")
    f.write("=" * 56 + "\n")
    f.write(f"{'Image':<10} {'µm/px':>12} {'Computed mag':>16} {'Probable mag':>14}\n")
    f.write("-" * 52 + "\n")
    for name, r in results.items():
        f.write(f"{name:<10} {r['pixel_size_um']:>12.4f} "
              f"{r['computed_mag']:>14.1f}x {r['probable_mag']:>12}x\n")

# Save results dictionary
with open('results.json', 'w') as f:
    json.dump(results, f, indent=4)

  CamIm01
  Lines detected      : 6
  Line centers y (px) : [ 61.2 110.9 160.5 210.  259.6 309.2]
  All gaps (px)       : [49.66 49.56 49.59 49.55 49.63]
  Major gaps (px)     : [49.66 49.56 49.59 49.55 49.63]
  Median major gap    : 49.59 px
  Pixel size          : 0.002017 mm/px
  Pixel size          : 2.0166 µm/px

  CamIm02
  Lines detected      : 2
  Line centers y (px) : [ 47.2 245.4]
  All gaps (px)       : [198.23]
  Major gaps (px)     : [198.23]
  Median major gap    : 198.23 px
  Pixel size          : 0.000504 mm/px
  Pixel size          : 0.5045 µm/px

  CamIm03
  Lines detected      : 2
  Line centers y (px) : [ 60.4 311.9]
  All gaps (px)       : [251.44]
  Major gaps (px)     : [251.44]
  Median major gap    : 251.44 px
  Pixel size          : 0.000398 mm/px
  Pixel size          : 0.3977 µm/px

                Table 1: Pixel Sizes                
Image          Gap (px)          mm/px          µm/px
--------------------------------------------------
CamIm01           49